# RITINI: Inferring Dynamic Regulatory Interaction Graphs from Time Series Data with Perturbations 

Prerequisites:
- Trained MIOFlow and decoded trajectories back to gene space.

In this notebook we will:
- Run RITINI to infer gene dynamics in gene regulatory networks

# Import libraries, set path and device

In [1]:
import os, sys, json, pickle, itertools, numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
import dgl
from gode.utils import get_device
from gode.data import make_train_test_dataframe
from ritini_module import ritini

In [2]:
device = get_device()

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

seed = 3
torch.manual_seed(seed)
np.random.seed(seed)

# Load and Preprocess Dataset

Load the dataset (output from MIOFlow) of shape ((n_timepoints, n_trajectrories, n_genes))

In [3]:
exp_dir = '/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/granger/granger_integrated/'
flow_name = '4_0_1'
subset_name = 'integrated'

In [4]:
"""
Load MIOFlow inferred trajectories.
Shape (n_timepoints, n_trajectrories, n_genes)
"""
trajectories = np.load(f"/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/dimchanger/dimchanger_{subset_name}/{flow_name}_gene_sp.npy", allow_pickle=True)

"""
Load gene names for plotting purposes
Shape (n_genes,)
"""
genes = np.load(f'/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/granger/granger_{subset_name}/{flow_name}_gene_names.npy', allow_pickle=True)

"""
Annotations are clusters of trajectories for a specific lineage. Here we assume just one set of trajectories. 
All labeled as 0.
"""
annotations = np.zeros(trajectories.shape[0])

"""
For simplicity, we will just train on the mean trajectory.
"""
mean_trajectories = trajectories.mean(axis=1, keepdims=True)

traj_data = {
    'trajectories': mean_trajectories,
    'genes': genes, 
    'annotations': annotations
}

In [5]:
"""
Load Granger causality data to construct the prior graph.
Shape (n_genes, n_genes)
"""
granger_df_all_T = pd.read_csv(exp_dir + f'{flow_name}_TF_only_signed_score.csv', index_col=0)
preds = granger_df_all_T.to_numpy()
tf_count = preds.shape[0]

top_genes_ranked = pd.read_csv(f"/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/granger/granger_{subset_name}/{flow_name}_total_scores_all.csv", index_col='source')
top_genes_ranked = top_genes_ranked.sort_values(by='top10', ascending=False)
order = top_genes_ranked.index
order = order[:tf_count]
top_genes = order

top_data = granger_df_all_T.loc[order, order].to_numpy()
top_data = np.abs(top_data)

# Creating Prior Graph
- Each node is a gene and the edges is the interaction between the genes.

In [6]:
# Create a directed graph from the adjacency matrix
G = nx.DiGraph()

# Add nodes with labels
for i, label in enumerate(top_genes):
    G.add_node(i, label=label)

# Add edges based on the adjacency matrix
for i in range(top_data.shape[0]):
    for j in range(top_data.shape[1]):
        if top_data[i, j] >= 5.:
            G.add_edge(i, j)

""" 
Convert the NetworkX graph to a DGL graph
"""
edges = list(G.edges())
u, v = np.array(edges).T
u = torch.IntTensor(u)
v = torch.IntTensor(v)

g = dgl.graph((u, v))
num_edges = g.number_of_edges()

g.ndata['feat'] = torch.Tensor(np.ones((len(g.nodes()), 1)))

#For plotting purposes.
ref_g = g.to_networkx()
ref_pos = nx.spring_layout(ref_g.to_undirected(), seed=seed)

for idx, node in enumerate(ref_g.nodes()):
    ref_g.nodes[idx]['color'] = plt.get_cmap('viridis', len(top_genes))(idx)
    ref_g.nodes[idx]['label'] = top_genes[idx]

In [7]:
""" 
Create the adjacency matrix.
"""
adjacency_matrix = nx.adjacency_matrix(G)
adjacency_matrix_negative = 1 - adjacency_matrix.todense() - np.eye(g.number_of_nodes())

In [8]:
"""
Split edges into training and testing sets for link prediction loss.
"""
# Mapping for edge ids
edge_ids = np.arange(g.number_of_edges())

# Shuffle
edge_ids = np.random.permutation(edge_ids)

test_size_percent = 30
test_size_fraction = test_size_percent / 100

edge_test_size = int(len(edge_ids) * test_size_fraction)
edge_train_size = g.number_of_edges() - edge_test_size

edge_test_pos_u = u[edge_ids[:edge_test_size]]
edge_test_pos_v = v[edge_ids[:edge_test_size]]

edge_train_pos_u = u[edge_ids[edge_test_size:]]
edge_train_pos_v = v[edge_ids[edge_test_size:]]

neg_u, neg_v = np.where(adjacency_matrix_negative != 0)
neg_edge_ids = np.random.choice(len(neg_u), g.number_of_edges())

edge_test_neg_u = neg_u[neg_edge_ids[:edge_test_size]]
edge_test_neg_v = neg_v[neg_edge_ids[:edge_test_size]]

edge_train_neg_u = neg_u[neg_edge_ids[edge_test_size:]]
edge_train_neg_v = neg_v[neg_edge_ids[edge_test_size:]]

# Train RiTINI
- Saves plots of the ground truth dynamics vs predicted dynamics
- Saves the inferred graph (gene regulatory network)

In [9]:
gene_subset_indices = np.where(np.isin(traj_data['genes'], top_genes))[0]
cell_subset_indices = np.random.choice(traj_data['trajectories'].shape[1], traj_data['trajectories'].shape[1], replace=False)

trajs = traj_data['trajectories']
trajs = trajs[::3]
trajs = trajs[:, cell_subset_indices]
trajs = trajs[:, :, gene_subset_indices]
traj_f = trajs.reshape(-1, trajs.shape[2])

In [10]:
pseudotimes = np.linspace(0, 1, trajs.shape[0])

In [11]:
annot_repeated = np.repeat(traj_data['annotations'][cell_subset_indices], trajs.shape[0])
pt_repeated = np.tile(pseudotimes, trajs.shape[1])
df = pd.DataFrame(traj_f, columns=top_genes, index=[f'cell_{i}' for i in range(traj_f.shape[0])])
df['pseudotime'] = pt_repeated

df['cell_types'] = [f'cell_type_{a}' for a in annot_repeated]
num_cell_types = len(df['cell_types'].unique())

In [12]:
df_train, df_test = make_train_test_dataframe(df)

In [13]:
n_cells_at_t = df['pseudotime'].value_counts()[0]

time_bins = np.sort(df.pseudotime.unique())
cell_types = np.sort(df.cell_types.unique())

t0, *_, tn = time_bins
time_tensor = torch.Tensor(time_bins)#.to(device)

in_feats = cell_types.size * n_cells_at_t
out_feats = cell_types.size * n_cells_at_t

In [14]:
# Create RITINI instance first
graph_trainer = ritini.RITINI(g, in_feats, out_feats, device)
# Initialize the model
model = graph_trainer.model
device = 'cpu'
model = model.to(device)
# Call the train_test method on the instance
train_g, train_pos_g, train_neg_g, test_pos_g, test_neg_g = graph_trainer.train_test(
    edge_ids, edge_train_pos_u, edge_train_pos_v, edge_train_neg_u, edge_train_neg_v, 
    edge_test_pos_u, edge_test_pos_v, edge_test_neg_u, edge_test_neg_v, edge_test_size
)

In [15]:
""" 
Hyperparameters for training 
"""
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1, weight_decay=5e-4)
scheduler = StepLR(optimizer, step_size=350, gamma=0.1)
criterion = torch.nn.MSELoss()

steps = 100
verbose_step = 1

lambda_l1 = 10
add_n = 5
del_n = 5
link_step = 2
sample_size = 10

In [ ]:
graph_trainer.train_loop(
        model, optimizer, scheduler, criterion, top_genes,
        train_g, train_pos_g, train_neg_g, test_pos_g, test_neg_g, 
        df_train, n_cells_at_t, time_bins, steps, link_step, add_n, del_n,
        verbose_step, num_cell_types, cell_types, ref_pos, ref_g)